In [ ]:
import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import torch.nn as nn
import torch.nn.functional as F


In [3]:
#Define transformations for the dataset
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

#Load the MNIST dataset
train_dataset = datasets.MNIST(root='./data', train=True, download=True, transform=transform)
test_dataset = datasets.MNIST(root='./data', train=False, download=True, transform=transform)

100.0%
100.0%
100.0%
100.0%


In [4]:
# Create data loaders
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

print(f"Training Dataset Size: {len(train_dataset)}")
print(f"Test Dataset Size: {len(test_dataset)}")

Training Dataset Size: 60000
Test Dataset Size: 10000


In [6]:
#Define the Model Architecture
class SimpleNN(nn.Module):
    def __init__(self):
        super(SimpleNN, self).__init__()
        self.flatten = nn.Flatten()
        self.fc1 = nn.Linear(28*28, 128)  
        self.fc2 = nn.Linear(128, 64)     
        self.fc3 = nn.Linear(64, 10)      

    def forward(self, x):
        x=self.flatten(x)
        x = F.relu(self.fc1(x)) 
        x = F.relu(self.fc2(x)) 
        x = self.fc3(x)            
        return x
    
# Instantiate the model, define the loss function and the optimizer
model = SimpleNN()
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)  



In [8]:
# Training loop
num_epochs = 10
for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    for images, labels in train_loader:
        optimizer.zero_grad()  
        outputs = model(images)  
        loss = criterion(outputs, labels)  
        loss.backward()        
        optimizer.step()       
        running_loss += loss.item()
    
    avg_loss = running_loss / len(train_loader)
    print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {avg_loss:.4f}")

Epoch [1/10], Loss: 0.4136
Epoch [2/10], Loss: 0.2005
Epoch [3/10], Loss: 0.1463
Epoch [4/10], Loss: 0.1199
Epoch [5/10], Loss: 0.1019
Epoch [6/10], Loss: 0.0910
Epoch [7/10], Loss: 0.0826
Epoch [8/10], Loss: 0.0724
Epoch [9/10], Loss: 0.0672
Epoch [10/10], Loss: 0.0592


In [9]:
# Evaluation Loop
model.eval()
correct = 0
total = 0
with torch.no_grad():  
    for images, labels in test_loader:
        outputs = model(images)  
        _, predicted = torch.max(outputs.data, 1)  
        total += labels.size(0)
        correct += (predicted == labels).sum().item()   
print(f"Test Accuracy: {100 * correct / total:.2f}%")


Test Accuracy: 97.28%


In [10]:
#Svave the model
torch.save(model.state_dict(), 'mnist_simple_nn.pth')

In [11]:
# Reload the model
loaded_model = SimpleNN()
loaded_model.load_state_dict(torch.load('mnist_simple_nn.pth'))
loaded_model.eval()

SimpleNN(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (fc1): Linear(in_features=784, out_features=128, bias=True)
  (fc2): Linear(in_features=128, out_features=64, bias=True)
  (fc3): Linear(in_features=64, out_features=10, bias=True)
)

In [ ]:
# update optimizer with a new learning rate
optimizer = torch.optim.Adam(loaded_model.parameters(), lr=0.0001)  
